# PG-MAP — Reproduce paper figures from scratch

**Preprint · under review at NeurIPS 2026** · [github.com/sophialanlan/PG-MAP](https://github.com/sophialanlan/PG-MAP)

This notebook regenerates the visual figures of the paper end-to-end using the published PG-MAP pipelines:

| Section | Paper figure | What it shows |
|---|---|---|
| §1 | Fig 1 — *pgmap_showcase* | Side-by-side SDXL **baseline vs PG-MAP** on 4 prompts (phoenix, swordsman, sailboat, turtle). The headline visual claim of the paper. |
| §2 | Fig 2 — *method_overview* (a) + (b) | DDIM **trajectory strips** at steps 0/10/20/30/40/49: MAP-c on the red-panda prompt (c-side rebinding), Reward-z on the galaxy-teacup prompt (z-side texture lift). |
| §3 | Fig 2 — *method_overview* (c) + (d) | Per-step **‖Δc‖** and **‖Δz‖** norms — the asymmetric, schedule-adaptive prior design surfaces as opposite slopes. |

**Compute budget**
- 4 SDXL generations for §1 + 4 for §2-trajectory + 2 for §3-norms ≈ **10 SDXL runs at 1024²**.
- On an H200 / A100: ~6 minutes total. On a free Colab T4: ~25 minutes (set `LITE_MODE=True` below to halve the runtime).
- Peak VRAM: ~24 GB (SDXL fp16 + PG-MAP reward backward). T4 (16 GB) needs `LITE_MODE=True`.

> 💡 **Open in Colab:** [colab.research.google.com/github/sophialanlan/PG-MAP/blob/main/notebooks/reproduce_paper_figures.ipynb](https://colab.research.google.com/github/sophialanlan/PG-MAP/blob/main/notebooks/reproduce_paper_figures.ipynb)

## 0. Setup

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q pgmap-align==1.5.2
!pip install -q matplotlib

In [ ]:
import os, gc, time, sys
from dataclasses import replace
from pathlib import Path
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import pgmap
from pgmap import sdxl_defaults, FrozenRewardModel
from pgmap_config import PGMAPConfig, PriorConfig, RefinementConfig, RewardConfig, baseline_config
from pgmap_sdxl import generate_sdxl_pgmap, generate_sdxl_baseline, load_sdxl_models

print('PG-MAP', pgmap.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# === Toggles ===
LITE_MODE = False        # True = halve steps + show only 2 of 4 showcase prompts (T4-friendly)
OUT_DIR = Path('reproduced_figures')
OUT_DIR.mkdir(exist_ok=True)

SEED = 123
STEPS = 24 if LITE_MODE else 50
print(f'\nReproducing at: seed={SEED}, num_steps={STEPS}, LITE_MODE={LITE_MODE}')

In [ ]:
print('Loading SDXL (cached after first run)...')
t0 = time.time()
models = load_sdxl_models('stabilityai/stable-diffusion-xl-base-1.0', dtype=torch.float16)
reward = FrozenRewardModel('pickscore', device='cuda')
print(f'  loaded in {time.time()-t0:.1f}s')

## §1. Fig 1 — `pgmap_showcase` grid (baseline vs PG-MAP)

Four prompts, same seed within each pair. Top row = static SDXL baseline; bottom row = joint PG-MAP refinement. The paper's Fig 1 uses the same four images; the gallery on [the project page](https://github.com/sophialanlan/PG-MAP/tree/main/docs/site) shows them at higher resolution.

In [ ]:
# Prompts from the paper Fig 1 caption (compressed_9pg_20260505/MAP_neurips2026_augmented.tex)
SHOWCASE_PROMPTS = [
    ('phoenix',   'a phoenix rising from ashes, vivid orange and red feathers, dramatic lighting'),
    ('swordsman', 'a swordsman mid-leap slashing through a glowing magical barrier'),
    ('sailboat',  'an old sailboat sailing through a thunderstorm with massive lightning bolts overhead'),
    ('turtle',    'a sea turtle wearing a backpack made of corals, floating through a forest'),
]
if LITE_MODE:
    SHOWCASE_PROMPTS = SHOWCASE_PROMPTS[:2]

def sdxl_pgmap_default(seed):
    cfg = sdxl_defaults()
    cfg.num_steps = STEPS
    cfg.seed = seed
    cfg.guidance_scale = 5.0
    cfg.rho = 0.5
    cfg.prior = PriorConfig(sigma_c=1.0, gamma=1.0)
    cfg.refinement = RefinementConfig(K=2, eta_c=1e-3, eta_z=5e-3)
    cfg.reward = RewardConfig(lambda_reward=0.10, rho_Q=0.3, grad_norm_strategy='unit')
    cfg.optimize_c = cfg.optimize_z = cfg.use_reward = True
    return cfg

def sdxl_baseline_cfg(seed):
    cfg = baseline_config('sdxl')
    cfg.num_steps = STEPS
    cfg.seed = seed
    cfg.guidance_scale = 5.0
    return cfg

baseline_imgs, pgmap_imgs = [], []
for name, prompt in SHOWCASE_PROMPTS:
    print(f'\n[{name}] baseline...')
    t0 = time.time()
    img_b, _ = generate_sdxl_baseline(prompt, models=models, config=sdxl_baseline_cfg(SEED))
    print(f'  {time.time()-t0:.1f}s')
    print(f'[{name}] PG-MAP...')
    t0 = time.time()
    img_p, _ = generate_sdxl_pgmap(prompt, models=models, config=sdxl_pgmap_default(SEED), reward_model=reward)
    print(f'  {time.time()-t0:.1f}s')
    baseline_imgs.append(img_b); pgmap_imgs.append(img_p)

# Compose the grid
n = len(SHOWCASE_PROMPTS)
fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
if n == 1: axes = axes.reshape(2, 1)
for j, (name, prompt) in enumerate(SHOWCASE_PROMPTS):
    axes[0, j].imshow(baseline_imgs[j]); axes[0, j].set_title(f'baseline\n{name}', fontsize=10); axes[0, j].axis('off')
    axes[1, j].imshow(pgmap_imgs[j]);    axes[1, j].set_title(f'PG-MAP\n{name}',   fontsize=10); axes[1, j].axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1_pgmap_showcase.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT_DIR / "fig1_pgmap_showcase.png"}')

## §2. Fig 2 (a)+(b) — Trajectory strips

Six DDIM checkpoints at steps {0, 10, 20, 30, 40, 49}, top row = baseline trajectory, bottom row = MAP-c (panda) or Reward-z (galaxy). The PG-MAP defaults match `gen_fig3_trajectories.py` in the original codebase.

Panel (a): the static baseline commits to a generic *human* astronaut by step ~30 and never recovers the panda subject; **MAP-c** brings the panda back as `c` is refined every step.

Panel (b): same teacup composition both rows, but **Reward-z** produces richer galaxy swirl and more saturated nebula colors via the per-step latent refinement against PickScore.

In [ ]:
PANDA_PROMPT  = 'a cinematic photo of a red panda astronaut, ultra detailed, dramatic lighting'
GALAXY_PROMPT = 'a tea cup with a tiny galaxy swirling inside, ultra detailed, dramatic lighting'

def traj_baseline_cfg(seed):
    cfg = baseline_config('sdxl')
    cfg.num_steps = STEPS; cfg.seed = seed; cfg.guidance_scale = 5.0
    cfg.save_progress = True; cfg.save_every = 10
    return cfg

def traj_mapc_cfg(seed):
    """MAP-c: optimize c only (no reward, no z). Matches gen_fig3_trajectories.py."""
    cfg = sdxl_defaults()
    cfg.num_steps = STEPS; cfg.seed = seed; cfg.guidance_scale = 5.0; cfg.rho = 0.5
    cfg.prior = PriorConfig(sigma_c=1.0, gamma=1.0)
    cfg.refinement = RefinementConfig(K=1, eta_c=1e-3, eta_z=5e-3)
    cfg.reward = RewardConfig(lambda_reward=0.05, rho_Q=0.3, grad_norm_strategy='unit')
    cfg.optimize_c, cfg.optimize_z, cfg.use_reward = True, False, False
    cfg.save_progress = True; cfg.save_every = 10
    return cfg

def traj_rewardz_cfg(seed):
    """Reward-z: optimize z only, against frozen PickScore reward."""
    cfg = sdxl_defaults()
    cfg.num_steps = STEPS; cfg.seed = seed; cfg.guidance_scale = 5.0; cfg.rho = 0.5
    cfg.prior = PriorConfig(sigma_c=1.0, gamma=1.0)
    cfg.refinement = RefinementConfig(K=1, eta_c=1e-3, eta_z=5e-3)
    cfg.reward = RewardConfig(lambda_reward=0.05, rho_Q=0.3, grad_norm_strategy='unit')
    cfg.optimize_c, cfg.optimize_z, cfg.use_reward = False, True, True
    cfg.save_progress = True; cfg.save_every = 10
    return cfg

print('[panda] baseline trajectory...')
t0 = time.time(); _, logs_b_panda = generate_sdxl_baseline(PANDA_PROMPT, models=models, config=traj_baseline_cfg(SEED)); print(f'  {time.time()-t0:.1f}s')
print('[panda] MAP-c trajectory...')
t0 = time.time(); _, logs_mapc = generate_sdxl_pgmap(PANDA_PROMPT, models=models, config=traj_mapc_cfg(SEED), reward_model=None); print(f'  {time.time()-t0:.1f}s')
print('[galaxy] baseline trajectory...')
t0 = time.time(); _, logs_b_galaxy = generate_sdxl_baseline(GALAXY_PROMPT, models=models, config=traj_baseline_cfg(SEED)); print(f'  {time.time()-t0:.1f}s')
print('[galaxy] Reward-z trajectory...')
t0 = time.time(); _, logs_rewardz = generate_sdxl_pgmap(GALAXY_PROMPT, models=models, config=traj_rewardz_cfg(SEED), reward_model=reward); print(f'  {time.time()-t0:.1f}s')

In [ ]:
def render_trajectory_panel(logs_top, logs_bot, title_top, title_bot, savepath):
    """2x6 strip: top = baseline ckpts, bottom = PG-MAP variant ckpts."""
    top  = logs_top.get('mid_imgs', [])
    bot  = logs_bot.get('mid_imgs', [])
    n_cols = max(len(top), len(bot))
    if n_cols == 0:
        print('  (no mid_imgs — check save_progress in config)')
        return
    fig, axes = plt.subplots(2, n_cols, figsize=(2.6*n_cols, 5.4))
    if n_cols == 1: axes = axes.reshape(2, 1)
    for j in range(n_cols):
        if j < len(top):
            step_i, img = top[j]; axes[0, j].imshow(img); axes[0, j].set_title(f'step {step_i}', fontsize=9)
        axes[0, j].axis('off')
        if j < len(bot):
            step_i, img = bot[j]; axes[1, j].imshow(img)
        axes[1, j].axis('off')
    axes[0, 0].set_ylabel(title_top, fontsize=11, rotation=0, ha='right', va='center', labelpad=20)
    axes[1, 0].set_ylabel(title_bot, fontsize=11, rotation=0, ha='right', va='center', labelpad=20)
    # Re-enable y-axis for labels
    for ax in (axes[0, 0], axes[1, 0]):
        ax.tick_params(left=False, labelleft=False, bottom=False, labelbottom=False)
        for spine in ax.spines.values(): spine.set_visible(False)
        ax.set_axis_on()
    plt.tight_layout()
    plt.savefig(savepath, dpi=110, bbox_inches='tight')
    plt.show()
    print(f'Saved: {savepath}')

print('\nFig 2(a) — panda c-side trajectory (baseline misses panda; MAP-c rebinds)')
render_trajectory_panel(logs_b_panda, logs_mapc, 'baseline', 'MAP-c',
                          OUT_DIR / 'fig2a_panda_trajectory.png')

print('\nFig 2(b) — galaxy z-side trajectory (same composition, richer texture)')
render_trajectory_panel(logs_b_galaxy, logs_rewardz, 'baseline', 'Reward-z',
                          OUT_DIR / 'fig2b_galaxy_trajectory.png')

## §3. Fig 2 (c)+(d) — ‖Δc‖ and ‖Δz‖ norm curves

Re-runs the panda prompt with `save_c_traj=True` (records c at every denoising step) and the galaxy prompt with `save_z_traj=True` (records pre/post-refinement z_t pairs). Then computes the per-step norms.

The paper's qualitative prediction (with $K=2$, $50$ DDIM steps): under a **constant** $\sigma_c$, ‖Δc‖ grows toward the data end (cross-attention signal sharpens). Under the **schedule-adaptive** $\sigma_z(t)=\gamma\sqrt{1-\bar\alpha_t}$, ‖Δz‖ decays toward the data end as the trust region tightens. The exact shape depends on $K_\text{inner}$ and the step count; in `LITE_MODE` the trend may not be fully resolved.

In [ ]:
def traj_mapc_with_c_log(seed):
    cfg = traj_mapc_cfg(seed); cfg.save_c_traj = True; return cfg
def traj_rewardz_with_z_log(seed):
    cfg = traj_rewardz_cfg(seed); cfg.save_z_traj = True; return cfg

print('[panda] MAP-c with save_c_traj=True...')
_, logs_cnorm = generate_sdxl_pgmap(PANDA_PROMPT, models=models, config=traj_mapc_with_c_log(SEED), reward_model=None)
print('[galaxy] Reward-z with save_z_traj=True...')
_, logs_znorm = generate_sdxl_pgmap(GALAXY_PROMPT, models=models, config=traj_rewardz_with_z_log(SEED), reward_model=reward)

c_traj = logs_cnorm['c_traj']                   # (T, B, 77, 2048) for SDXL
c0 = c_traj[0]
delta_c = (c_traj - c0).reshape(c_traj.shape[0], -1).norm(dim=-1).detach().cpu().numpy()

z_before = logs_znorm['z_traj_before']          # (T, B, C, H, W)
z_after  = logs_znorm['z_traj_after']
delta_z  = (z_after - z_before).reshape(z_after.shape[0], -1).norm(dim=-1).detach().cpu().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(delta_c, color='tab:blue', linewidth=2)
ax1.set_xlabel('denoising step (high noise → data end)')
ax1.set_ylabel(r'$\|\Delta c\|_2$ (MAP-c only)')
ax1.set_title('Fig 2(c) — ‖Δc‖ grows toward the data end')
ax1.grid(alpha=0.3)
ax2.plot(delta_z, color='tab:red', linewidth=2)
ax2.set_xlabel('denoising step (high noise → data end)')
ax2.set_ylabel(r'$\|\Delta z\|_2$ (Reward-z only)')
ax2.set_title('Fig 2(d) — ‖Δz‖ decays as σ_z(t) tightens')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig2cd_delta_norms.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT_DIR / "fig2cd_delta_norms.png"}')
print(f'\nDelta-c values (first/last): {delta_c[0]:.3f} → {delta_c[-1]:.3f}')
print(f'Delta-z values (first/last): {delta_z[0]:.3f} → {delta_z[-1]:.3f}')

## Summary

Three figures regenerated from scratch in this notebook:

| File | Paper figure |
|---|---|
| `reproduced_figures/fig1_pgmap_showcase.png`     | Fig 1 — pgmap_showcase grid (baseline vs PG-MAP) |
| `reproduced_figures/fig2a_panda_trajectory.png`  | Fig 2 (a) — MAP-c c-side trajectory on the panda prompt |
| `reproduced_figures/fig2b_galaxy_trajectory.png` | Fig 2 (b) — Reward-z z-side trajectory on the galaxy prompt |
| `reproduced_figures/fig2cd_delta_norms.png`      | Fig 2 (c, d) — ‖Δc‖ growing / ‖Δz‖ decaying |

Cross-check against the paper's pre-generated images at [docs/site/images/](https://github.com/sophialanlan/PG-MAP/tree/main/docs/site/images). On identical hardware (RTX PRO 6000 Blackwell, fp16, seed 123) the regenerated images are pixel-exact; cross-GPU drift is within VAE precision (≤ 1/255 mean abs deviation).

## Citation

```bibtex
@misc{sun2026pgmap,
  title={{PG-MAP}: Joint {MAP} Optimization for Inference-Time Alignment of Diffusion and Flow-Matching Models},
  author={Sun, Ruolan and Polak, Pawel},
  year={2026},
  eprint={2606.22958},
  archivePrefix={arXiv},
  primaryClass={cs.LG},
  note={Under review at NeurIPS 2026},
  url={https://arxiv.org/abs/2606.22958}
}
```